## 커스텀 데이터셋을 만들어서 사용하기
###### 데이터로더 객체에 학습에 사용될 데이터 전체를 보관했다가 모델 학습을 할 때 배치 크기 만큼 데이터를 꺼내서 사용하는 방법
###### 데이터를 미리 잘라 놓는 것이 아니라 내부적으로 iterator(반복자)에 포함된 index를 이용하여 배치 크기만큼 데이터를 반환함

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [2]:
class CustomDataset(Dataset):
    def __init__(self, csv_file):
        self.label= pd.read_csv(csv_file)

    def __len__(self):
        return len(self.label)
    
    def __getitem__(self, idx):
        sample = torch.tensor(self.label.iloc[idx, 0:3]).int()
        label = torch.tensor(self.label.iloc[idx, 3]).int()
        return sample, label

In [3]:
import pandas as pd
print(pd.__file__)

c:\Users\USER\miniconda3\envs\hb\lib\site-packages\pandas\__init__.py


In [4]:
tensor_dataset = CustomDataset(r'C:\Users\USER\OneDrive\바탕 화면\my_study_python\mygit\PyTorch_study\딥러닝파이토치교과서_예제\chap02\data\car_evaluation.csv')
dataset = DataLoader(tensor_dataset, batch_size=4, shuffle=True) #데이터셋을 torch.utils.data.DataLoader로 감싸서 배치 단위로 데이터를 불러올 수 있도록 함

In [5]:
tensor_dataset.label

,price,maint,doors,persons,lug_capacity,safety,output
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,good
1724,low,low,5more,more,med,high,vgood
1725,low,low,5more,more,big,low,unacc
1726,low,low,5more,more,big,med,good


In [6]:
tensor_dataset.__len__()

1728

## 파이토치에서 제공하는 데이터셋 사용

In [7]:
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)

2.10.0+cpu
0.25.0+cpu


In [11]:
import torchvision.transforms as transforms

mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (1.0,)) # 평균이 0.5이고 표준편차가 1.0인 정규화
])

from torchvision.datasets import MNIST
import requests
download_root = r'C:\Users\USER\OneDrive\바탕 화면\my_study_python\mygit\PyTorch_study\dataset'

train_dataset = MNIST(root=download_root, train=True, transform=mnist_transform, download=True)
valid_dataset = MNIST(root=download_root, train=True, transform=mnist_transform, download=True)
test_dataset = MNIST(root=download_root, train=False, transform=mnist_transform, download=True)


100.0%
100.0%
100.0%
100.0%


## 모델 정의
#### 파이토치에서 모델을 정의하기 위해서는 모듈을 상속한 클래스를 사용
###### 계층(layer): 모듈 또는 모듈을 구성하는 한 개의 계층으로 합성곱층, 선형 계층 등이 있음
###### 모듈(module): 한 개 이상의 계층이 모여서 구성된 것으로, 모듈이 모여 새로운 모듈을 만들 수도 있음
###### 모델(model): 최종적으로 원하는 네트워크로, 한 개의 모듈이 모델이 될 수도 있음

#### nn.Module을 상속 받아 정의하는 방법

In [13]:
from torch.nn import Module, Linear, ReLU

class MLP(Module):
    def __init__(self, inputs):
        super(MLP, self).__init__()
        self.layer = Linear(inputs, 10) # 입력층에서 출력층으로 가는 선형 계층 정의
        self.activation = ReLU() # ReLU 활성화 함수 정의

    def forward(self, x):
        x = self.layer(x) # 선형 계층을 통과
        x = self.activation(x) # 활성화 함수를 통과
        return x

#### Sequential 신경망을 정의하는 방법

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1), # 입력 채널 수는 1 (흑백 이미지), 출력 채널 수는 32, 커널 크기는 3x3, 스트라이드는 1, 패딩은 1
            nn.ReLU()
            nn.MaxPool2d(kernel_size=2, stride=2) # 커널 크기는 2x2, 스트라이드는 2
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1), # 입력 채널 수는 32, 출력 채널 수는 64, 커널 크기는 3x3, 스트라이드는 1, 패딩은 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # 커널 크기는 2x2, 스트라이드는 2
        )
        self.layer3 = nn.Sequential(
            nn.Linear(in_features=64*7*7, out_features=128, bias=True), # 입력 특징 수는 64*7*7 (Conv 레이어의 출력 크기), 출력 특징 수는 128
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.layer1(x) # 첫 번째 레이어를 통과
        x = self.layer2(x) # 두 번째 레이어를 통과
        x = x.view(x.size(0), -1) # 텐서를 2차원으로 변환 (배치 크기, 특징 수)
        x = self.layer3(x) # 세 번째 레이어를 통과
        return x

### 모델의 훈련 및 평가 예시 코드

In [ ]:
from xml.parsers.expat import model
from torch.optim import optimizer

criterion = torch.nn.MSELoss
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9) # 모델의 매개변수를 최적화하기 위해 SGD 옵티마이저를 사용, 학습률은 0.01, 모멘텀은 0.9   
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, 
                                              lr_lambda=lambda epoch: 0.1 ** (epoch // 10)) # 학습률 스케줄러 정의, 10 에폭마다 학습률을 0.1배로 감소

# 모델 훈련 예시
for epoch in range(100):
    yhat = model(x_train) # 모델을 사용하여 예측값 계산
    loss = criterion(yhat, y_train) # 손실 함수 인스턴스 생성
    optimizer.zero_grad() # 오차가 중첩적으로 쌓이지 않도록 옵티마이저의 기울기를 초기화
    loss.backward() #  역전파 학습
    optimizer.step() # 모델의 매개변수를 업데이트

In [ ]:
# pip install torchmetrics
import torch
import torchmetrics

# 함수를 이용해 모델을 평가하는 코드
preds = torch.randn(10, 5).softmax(dim=1) # 예측값을 소프트맥스 함수로 변환하여 확률로 만듦
targets = torch.randint(0, 5, (10,)) # 0부터4까지의 정수로 이루어진 타깃 레이블 생성
accuracy = torchmetrics.Accuracy() # 정확도 계산을 위한 torchmetrics의 Accuracy 클래스 인스턴스 생성
accuracy(preds, targets) # 정확도 계산

In [ ]:
# 모듈을 이용해 모델을 평가하는 코드
import torch
import torchmetrics
metric = torchmetrics.Accuracy() # 정확도 계산을 위한 torchmetrics의 Accuracy 클래스 인스턴스 생성

n_batchs = 5
for i in range(n_batchs):
    preds = torch.randn(10, 5).softmax(dim=1) # 예측값을 소프트맥스 함수로 변환하여 확률로 만듦
    targets = torch.randint(0, 5, (10,)) # 0부터4까지의 정수로 이루어진 타깃 레이블 생성

    acc = metric(preds, targets) # 정확도 계산
    print(f'Batch {i+1}/{n_batchs} - Accuracy: {acc:.4f}')  # 각 배치마다 정확도를 계산하여 출력

acc = metric.compute() # 전체 배치에 대한 정확도 계산
print(f'Overall Accuracy: {acc:.4f}') # 전체 배치에 대한 정확도 출력

In [16]:
# pip install tensorboard
import torch
import torch.utils.tensorboard as tb
writer = tb.SummaryWriter() # TensorBoard SummaryWriter 인스턴스 생성

# 훈련 과정 모니터링
for epoch in range(num_epochs):
    # 훈련 코드 (예: 모델 훈련, 손실 계산 등)
    model.train() # 모델을 훈련 모드로 설정
    batch_loss = 0.0 

    for i, (x,y) in enumerate(train_loader):
        # 모델 훈련 코드 (예: 순전파, 손실 계산, 역전파, 매개변수 업데이트)
        optimizer.zero_grad() # 옵티마이저의 기울기를 초기화
        outputs = model(x) # 모델을 사용하여 예측값 계산
        loss = criterion(outputs, y) # 손실 함수 인스턴스 생성
        writer.add_scalar('Loss/train', loss.item(), epoch) # TensorBoard에 손실 값 기록   
        loss.backward() # 역전파 학습
        optimizer.step() # 모델의 매개변수를 업데이트

        batch_loss += loss.item() # 배치 손실 누적

writer.close() # SummaryWriter 닫기

NameError: name 'num_epochs' is not defined

In [ ]:
## TensorBoard 실행 명령어
# tesnsorboard --logdir=runs --port=6006

In [ ]:
# model.eval() 사용법
model.eval() # 모델을 평가 모드로 설정, 드롭아웃과 배치 정규화 레이어가 평가 모드로 전환되어 예측 시 일관된 결과를 제공
with torch.no_grad(): # 평가 시에는 기울기를 계산하지 않도록 설정하여 메모리 사용량과 계산 속도를 최적화
    valid_loss = 0.0

    for x, y in valid_dataloader:
        outputs = model(x) # 모델을 사용하여 예측값 계산
        # 평가 코드 (예: 정확도 계산 등)
        loss = F.cross_entropy(outputs, y.long().squeeze()) # 손실 함수 계산 (예: 교차 엔트로피 손실)
        valid_loss += float(loss) # 검증 손실 누적  
        y_hat += [outputs] # 예측값의 클래스 인덱스 저장

valid_loss /= len(valid_dataloader) # 검증 손실 평균 계산